In [1]:
# ============================================================
# ASSOCIATION PATTERN MINING - FIXED VERSION
# ============================================================

# Install mlxtend if needed
!pip install mlxtend -q


# ============================================================
# STEP 1: IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np

from mlxtend.frequent_patterns import apriori, association_rules


# ============================================================
# STEP 2: LOAD DATASET
# ============================================================

df = pd.read_csv(
    'phishing_website_feature_selected.csv'
)

print("=" * 70)
print("ASSOCIATION PATTERN MINING")
print("=" * 70)

print("\nOriginal dataset shape:", df.shape)


# ============================================================
# STEP 3: CHECK TARGET VALUES
# ============================================================

print("\nTarget distribution:")
print(df['phishing'].value_counts())

# Remove rows with missing target
df = df.dropna(subset=['phishing']).copy()

# Make sure target is integer
df['phishing'] = df['phishing'].astype(int)


# ============================================================
# STEP 4: SEPARATE FEATURES
# ============================================================

X = df.drop(columns=['phishing']).copy()


# ============================================================
# STEP 5: REMOVE CONSTANT FEATURES
# ============================================================

constant_features = [
    col for col in X.columns
    if X[col].nunique() <= 1
]

if constant_features:
    X = X.drop(columns=constant_features)

print("\nConstant features removed:", len(constant_features))


# ============================================================
# STEP 6: SELECT FEATURES FOR ASSOCIATION MINING
# ============================================================

# Select top 15 non-constant features with highest variance

feature_variance = X.var().sort_values(ascending=False)

top_features = feature_variance.head(15).index.tolist()

print("\nFeatures selected for association mining:")

for col in top_features:
    print("✓", col)


# ============================================================
# STEP 7: DISCRETIZE NUMERIC FEATURES
# ============================================================

print("\n" + "=" * 70)
print("DISCRETIZING FEATURES")
print("=" * 70)

transaction_df = pd.DataFrame(index=df.index)


for col in top_features:

    values = X[col].copy()

    unique_count = values.nunique()

    # --------------------------------------------------------
    # Case 1: Binary feature
    # --------------------------------------------------------

    if unique_count == 2:

        unique_values = sorted(values.unique())

        transaction_df[col] = np.where(
            values == unique_values[0],
            "Low",
            "High"
        )

    # --------------------------------------------------------
    # Case 2: Very few unique values
    # --------------------------------------------------------

    elif unique_count <= 5:

        transaction_df[col] = values.astype(str)

    # --------------------------------------------------------
    # Case 3: Continuous feature
    # --------------------------------------------------------

    else:

        try:

            transaction_df[col] = pd.qcut(
                values.rank(method='first'),
                q=3,
                labels=[
                    'Low',
                    'Medium',
                    'High'
                ]
            )

        except Exception as e:

            print(
                f"Warning: Could not discretize {col}"
            )

            median_value = values.median()

            transaction_df[col] = np.where(
                values <= median_value,
                "Low",
                "High"
            )


# ============================================================
# STEP 8: ADD TARGET
# ============================================================

transaction_df['phishing'] = np.where(
    df['phishing'] == 1,
    'Phishing',
    'Legitimate'
)


print("\nSample discretized data:")
print(transaction_df.head())


# ============================================================
# STEP 9: CHECK VALUE DISTRIBUTION
# ============================================================

print("\n" + "=" * 70)
print("DISCRETIZATION CHECK")
print("=" * 70)

for col in top_features[:5]:

    print(f"\n{col}:")

    print(
        transaction_df[col]
        .value_counts()
    )


# ============================================================
# STEP 10: ONE-HOT ENCODE TRANSACTION DATA
# ============================================================

print("\n" + "=" * 70)
print("CREATING TRANSACTION MATRIX")
print("=" * 70)

transaction_encoded = pd.get_dummies(
    transaction_df,
    prefix_sep="="
).astype(bool)


print("\nTransaction matrix shape:")
print(transaction_encoded.shape)

print("\nSample columns:")
print(
    transaction_encoded.columns.tolist()[:20]
)


# ============================================================
# STEP 11: FIND FREQUENT ITEMSETS
# ============================================================

print("\n" + "=" * 70)
print("FINDING FREQUENT ITEMSETS")
print("=" * 70)

frequent_itemsets = apriori(
    transaction_encoded,
    min_support=0.05,
    use_colnames=True,
    max_len=3
)

frequent_itemsets['length'] = (
    frequent_itemsets['itemsets']
    .apply(len)
)

print(
    "\nFrequent itemsets found:",
    len(frequent_itemsets)
)

print("\nTop frequent itemsets:")

print(
    frequent_itemsets
    .sort_values(
        by='support',
        ascending=False
    )
    .head(10)
)


# ============================================================
# STEP 12: GENERATE ASSOCIATION RULES
# ============================================================

print("\n" + "=" * 70)
print("GENERATING ASSOCIATION RULES")
print("=" * 70)

rules = association_rules(
    frequent_itemsets,
    metric='confidence',
    min_threshold=0.60
)


# ============================================================
# IMPORTANT: CHECK RULE COLUMNS
# ============================================================

print("\nAvailable rule columns:")

print(
    rules.columns.tolist()
)


# Check required columns

required_columns = [
    'antecedents',
    'consequents',
    'support',
    'confidence',
    'lift'
]

missing_columns = [
    col for col in required_columns
    if col not in rules.columns
]

if missing_columns:

    print(
        "\nERROR: Missing columns:",
        missing_columns
    )

    print(
        "\nAssociation rules could not be generated correctly."
    )

else:

    print(
        "\nTotal association rules found:",
        len(rules)
    )


    # ========================================================
    # HELPER FUNCTION
    # ========================================================

    def contains_item(itemset, target_item):

        return any(
            str(item) == target_item
            for item in itemset
        )


    # ========================================================
    # PATTERN 1: PHISHING ASSOCIATION
    # ========================================================

    print("\n" + "=" * 70)
    print("PATTERN 1: PHISHING ASSOCIATION PATTERNS")
    print("=" * 70)


    phishing_rules = rules[
        rules['consequents'].apply(
            lambda x: contains_item(
                x,
                'phishing=Phishing'
            )
        )
    ].copy()


    # Remove rules where phishing is also in antecedent

    phishing_rules = phishing_rules[
        ~phishing_rules['antecedents'].apply(
            lambda x: contains_item(
                x,
                'phishing=Phishing'
            )
        )
    ]


    print(
        "\nPhishing rules found:",
        len(phishing_rules)
    )


    if len(phishing_rules) > 0:

        phishing_rules = (
            phishing_rules
            .sort_values(
                by=[
                    'lift',
                    'confidence'
                ],
                ascending=False
            )
        )


        print(
            phishing_rules[
                [
                    'antecedents',
                    'consequents',
                    'support',
                    'confidence',
                    'lift'
                ]
            ]
            .head(10)
        )

    else:

        print(
            "\nNo phishing rules found."
        )


    # ========================================================
    # PATTERN 2: LEGITIMATE ASSOCIATION
    # ========================================================

    print("\n" + "=" * 70)
    print("PATTERN 2: LEGITIMATE ASSOCIATION PATTERNS")
    print("=" * 70)


    legitimate_rules = rules[
        rules['consequents'].apply(
            lambda x: contains_item(
                x,
                'phishing=Legitimate'
            )
        )
    ].copy()


    legitimate_rules = legitimate_rules[
        ~legitimate_rules['antecedents'].apply(
            lambda x: contains_item(
                x,
                'phishing=Legitimate'
            )
        )
    ]


    print(
        "\nLegitimate rules found:",
        len(legitimate_rules)
    )


    if len(legitimate_rules) > 0:

        legitimate_rules = (
            legitimate_rules
            .sort_values(
                by=[
                    'lift',
                    'confidence'
                ],
                ascending=False
            )
        )


        print(
            legitimate_rules[
                [
                    'antecedents',
                    'consequents',
                    'support',
                    'confidence',
                    'lift'
                ]
            ]
            .head(10)
        )

    else:

        print(
            "\nNo legitimate rules found."
        )


    # ========================================================
    # PATTERN 3: FEATURE-TO-FEATURE ASSOCIATION
    # ========================================================

    print("\n" + "=" * 70)
    print("PATTERN 3: FEATURE-TO-FEATURE ASSOCIATION")
    print("=" * 70)


    def contains_target(itemset):

        return any(
            str(item).startswith('phishing=')
            for item in itemset
        )


    feature_rules = rules[
        ~rules['antecedents'].apply(
            contains_target
        )
        &
        ~rules['consequents'].apply(
            contains_target
        )
    ].copy()


    print(
        "\nFeature association rules found:",
        len(feature_rules)
    )


    if len(feature_rules) > 0:

        feature_rules = (
            feature_rules
            .sort_values(
                by=[
                    'lift',
                    'confidence'
                ],
                ascending=False
            )
        )


        print(
            feature_rules[
                [
                    'antecedents',
                    'consequents',
                    'support',
                    'confidence',
                    'lift'
                ]
            ]
            .head(10)
        )

    else:

        print(
            "\nNo feature association rules found."
        )


    # ========================================================
    # STEP 13: SELECT 3 FINAL PATTERNS
    # ========================================================

    print("\n" + "=" * 70)
    print("FINAL 3 ASSOCIATION PATTERNS")
    print("=" * 70)


    pattern_candidates = []


    # Best phishing pattern

    if len(phishing_rules) > 0:

        pattern_candidates.append(
            (
                "PHISHING PATTERN",
                phishing_rules.iloc[0]
            )
        )


    # Best legitimate pattern

    if len(legitimate_rules) > 0:

        pattern_candidates.append(
            (
                "LEGITIMATE PATTERN",
                legitimate_rules.iloc[0]
            )
        )


    # Best feature pattern

    if len(feature_rules) > 0:

        pattern_candidates.append(
            (
                "FEATURE ASSOCIATION PATTERN",
                feature_rules.iloc[0]
            )
        )


    for i, (
        pattern_type,
        rule
    ) in enumerate(
        pattern_candidates,
        start=1
    ):

        print(
            f"\n{'-' * 50}"
        )

        print(
            f"PATTERN {i}: "
            f"{pattern_type}"
        )

        print(
            f"{'-' * 50}"
        )

        print(
            "\nIF:"
        )

        print(
            list(
                rule['antecedents']
            )
        )

        print(
            "\nTHEN:"
        )

        print(
            list(
                rule['consequents']
            )
        )

        print(
            f"\nSupport: "
            f"{rule['support']:.4f}"
        )

        print(
            f"Confidence: "
            f"{rule['confidence']:.4f}"
        )

        print(
            f"Lift: "
            f"{rule['lift']:.4f}"
        )


    # ========================================================
    # STEP 14: SAVE RESULTS
    # ========================================================

    rules.to_csv(
        'all_association_rules.csv',
        index=False
    )


    phishing_rules.to_csv(
        'phishing_association_patterns.csv',
        index=False
    )


    legitimate_rules.to_csv(
        'legitimate_association_patterns.csv',
        index=False
    )


    feature_rules.to_csv(
        'feature_association_patterns.csv',
        index=False
    )


    print("\n" + "=" * 70)
    print("ASSOCIATION MINING COMPLETED")
    print("=" * 70)

    print(
        "\nFiles saved successfully."
    )


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
ASSOCIATION PATTERN MINING

Original dataset shape: (9944, 25)

Target distribution:
phishing
0    6491
1    3453
Name: count, dtype: int64

Constant features removed: 0

Features selected for association mining:
✓ url_shortened
✓ domain_in_ip
✓ qty_at_url
✓ time_domain_expiration
✓ length_url
✓ domain_spf
✓ qty_redirects
✓ qty_mx_servers
✓ qty_nameservers
✓ qty_ip_resolved
✓ time_domain_activation
✓ asn_ip
✓ email_in_url
✓ qty_dot_url
✓ qty_hyphen_url

DISCRETIZING FEATURES

Sample discretized data:
  url_shortened domain_in_ip           qty_at_url time_domain_expiration  \
0           Low          Low  -0.1455479398857974                 Medium   
1           Low          Low  -0.1455479398857974                    Low   
2           Low          Low  -0.1455479398857974                 Medium   
3           Low          Low  -0.1455479398857974                 Medium   
4 